In [7]:
import json
import os
from io import BytesIO
from urllib.request import Request, urlopen

import pandas as pd

api_url = os.getenv("CORE_API_URL", "http://127.0.0.1:8000")

payload = {
    "start_date": "2020-01-01",
    "end_date": "2026-01-01",
    "codes": [],
    "factors": ["weight_000300SH"],
    "derivatives": {
        "is_member": {
            "type": "DIRECT",
            "op": "binary.gt",
            "fields": {
                "left": "weight_000300SH",
                "right": 0,
            },
            "params": {},
        }
    },
    "filters": ["is_member"],
}

request = Request(
    f"{api_url}/query",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urlopen(request, timeout=600) as response:
    components = pd.read_parquet(BytesIO(response.read()))

components = components.sort_values(["time", "code"]).reset_index(drop=True)
components

,time,code,weight_000300SH,is_member
0,2025-01-02,000001.SZ,0.551,True
1,2025-01-02,000002.SZ,0.240,True
2,2025-01-02,000063.SZ,0.631,True
3,2025-01-02,000100.SZ,0.458,True
4,2025-01-02,000157.SZ,0.174,True
...,...,...,...,...
72895,2025-12-31,688303.SH,0.068,True
72896,2025-12-31,688396.SH,0.111,True
72897,2025-12-31,688472.SH,0.087,True
72898,2025-12-31,688506.SH,0.106,True


In [8]:
codes = components["code"].drop_duplicates().tolist()

payload = {
    "start_date": "2020-01-01",
    "end_date": "2026-01-01",
    "codes": codes,
    "factors": [],
    "derivatives": {
        "eligible": {
            "type": "DIRECT",
            "op": "multiary.and",
            "fields": {
                "cols": [
                    {
                        "type": "DIRECT",
                        "op": "binary.gt",
                        "fields": {
                            "left": "weight_000300SH",
                            "right": 0,
                        },
                        "params": {},
                    },
                    {
                        "type": "DIRECT",
                        "op": "binary.eq",
                        "fields": {"left": "is_st", "right": 0},
                        "params": {},
                    },
                    {
                        "type": "DIRECT",
                        "op": "binary.gt",
                        "fields": {"left": "pe", "right": 0},
                        "params": {},
                    },
                    {
                        "type": "DIRECT",
                        "op": "binary.lt",
                        "fields": {"left": "pe", "right": 50},
                        "params": {},
                    },
                    {
                        "type": "DIRECT",
                        "op": "binary.gt",
                        "fields": {"left": "circ_mv", "right": 0},
                        "params": {},
                    },
                    {
                        "type": "DIRECT",
                        "op": "unary.not_null",
                        "fields": {"col": "close"},
                        "params": {},
                    },
                ]
            },
            "params": {},
        },
        "composite_factor": {
            "type": "CS",
            "op": "unary.zscore",
            "fields": {
                "col": {
                    "type": "CS",
                    "op": "controls.neutralize_by",
                    "fields": {
                        "target": {
                            "type": "TS",
                            "op": "unary.pct_change",
                            "fields": {"col": "close"},
                            "params": {"periods": 20},
                        },
                        "controls": [
                            {
                                "type": "DIRECT",
                                "op": "unary.log",
                                "fields": {"col": "circ_mv"},
                                "params": {},
                            }
                        ],
                    },
                    "params": {"intercept": True},
                    "on": "eligible",
                }
            },
            "params": {"ddof": 0},
            "on": "eligible",
        },
        "selected": {
            "type": "DIRECT",
            "op": "unary.not_null",
            "fields": {"col": "composite_factor"},
            "params": {},
        },
    },
    "filters": ["eligible", "selected"],
}

request = Request(
    f"{api_url}/query",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urlopen(request, timeout=600) as response:
    factors = pd.read_parquet(BytesIO(response.read()))

factors = factors[["time", "code", "composite_factor"]]
factors = factors.sort_values(["time", "code"]).reset_index(drop=True)
factors

,time,code,composite_factor
0,2025-02-07,000001.SZ,-0.194418
1,2025-02-07,000002.SZ,0.318762
2,2025-02-07,000063.SZ,1.536243
3,2025-02-07,000100.SZ,0.144159
4,2025-02-07,000157.SZ,-0.187934
...,...,...,...
52249,2025-12-31,688036.SH,-0.770721
52250,2025-12-31,688169.SH,-0.662639
52251,2025-12-31,688187.SH,0.848184
52252,2025-12-31,688472.SH,-0.550097
